# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prasannadolas/prsna-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row = one content item, on one day, for one client (table fact_content_daily_performance).
Window: March 2026 (month=2026-03), a mid-panel month. Features use days 1-21 (the decision moment is the end of day 21). The label uses days 22-31. The final month (June 2026) stays sealed as a test month and is never touched here.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb scikit-learn scipy
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH, CUT = "2026-03", "2026-03-21"
src = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

# Setup only: look at the real column names, then fix the config below to match
print(con.sql(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]])

# CONFIG: edit these names to match the DESCRIBE output above
DATE, CLIENT, CONTENT = "report_date", "client_id", "content_id"
CLICKS, IMPR, POS = "clicks", "impressions", "avg_position"
AVAIL = "gsc_available"   # the boolean availability flag in your schema

con.execute(f"CREATE OR REPLACE TABLE m AS SELECT * FROM {src} WHERE month = '{MONTH}'")
print("loaded month:", MONTH)

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

loaded month: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (all from days 1-21 only): clicks_21d, impressions_21d, avg_position_21d, active_days_21d, clicks_days15_21.
LABEL: clicks in days 22-31 (label_clicks_next10d), modelled as log1p(clicks). This is a proxy for future performance.
CONTEXT (used to filter or split, not as features): the date column, the client column (used for the train/test split by client), the content column (used to group rows), and the availability flag (rows kept only where it IS TRUE).
EXCLUDED, with reasons:
- Any value from after day 21 in the features: that is the future, so it would leak the label.
- The final month (June 2026): sealed test month, reserved for later modelling weeks.
- The id columns as model inputs: pseudonymized keys carry no usable meaning and could let the model memorise identities.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

buckets = {
    "feature (days 1-21 only)": [CLICKS, IMPR, POS],
    "label (days 22-31)":       [CLICKS],
    "context":                  [DATE, CLIENT, CONTENT, AVAIL],
}
have = set(con.sql("SELECT * FROM m LIMIT 0").df().columns)
for bucket, cols in buckets.items():
    missing = [c for c in cols if c not in have]
    print(f"{bucket:28s} {cols}  ->  {'OK' if not missing else 'MISSING: ' + str(missing)}")


feature (days 1-21 only)     ['clicks', 'impressions', 'avg_position']  ->  MISSING: ['clicks', 'impressions', 'avg_position']
label (days 22-31)           ['clicks']  ->  MISSING: ['clicks']
context                      ['report_date', 'client_id', 'content_id', 'gsc_available']  ->  MISSING: ['client_id', 'content_id', 'gsc_available']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim above has a query below. Three verification queries: (1) grain, (2) row count and date span, (3) availability checked with IS TRUE. Then the five-feature frame and the deliberate-leak experiment.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema = con.sql("DESCRIBE m").df()[["column_name", "column_type"]]
print(schema.to_string(index=False))
names = list(schema.column_name)

def pick(*keys):
    for n in names:
        if all(k in n.lower() for k in keys):
            return n

DATE    = pick("date")
CLIENT  = pick("client", "hash") or pick("client")
CONTENT = pick("content", "hash") or pick("content")
CLICKS  = pick("click")
IMPR    = pick("impression")
POS     = pick("position")
bools   = list(schema[schema.column_type == "BOOLEAN"].column_name)
AVAIL   = next((b for b in bools if "avail" in b.lower()), None)

print("\nChosen:", dict(DATE=DATE, CLIENT=CLIENT, CONTENT=CONTENT,
                        CLICKS=CLICKS, IMPR=IMPR, POS=POS, AVAIL=AVAIL))
print("Boolean columns:", bools)

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

In [23]:
# Query 2: slice size and date span
con.sql(f"""
SELECT COUNT(*) AS n_rows, MIN({DATE}) AS first_day, MAX({DATE}) AS last_day,
       COUNT(DISTINCT {CLIENT}) AS n_clients, COUNT(DISTINCT {CONTENT}) AS n_content
FROM m""").df()

,n_rows,first_day,last_day,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [24]:
# Query 3: availability. How many rows survive IS TRUE, and how many are NULL
con.sql(f"""
SELECT COUNT(*) AS all_rows,
       COUNT_IF({AVAIL} IS TRUE) AS available_rows,
       COUNT_IF({AVAIL} IS NULL) AS null_rows,
       ROUND(100.0 * COUNT_IF({AVAIL} IS TRUE) / COUNT(*), 1) AS pct_survive
FROM m""").df()

,all_rows,available_rows,null_rows,pct_survive
0,9841378,3611061.0,0.0,36.7


Five features, each knowable at the decision moment (end of day 21):
- clicks_21d: sums clicks from days 1-21 only, all in the past at the cut-off.
- impressions_21d: sums impressions from the same past window.
- avg_position_21d: averages position over days 1-21, so no future day is included.
- active_days_21d: counts days with impressions up to day 21, fully observed by then.
- clicks_days15_21: the last week of the window, finished by the cut-off, so it shows the recent direction.

In [25]:
feat = con.sql(f"""
WITH a AS (SELECT * FROM m WHERE {AVAIL} IS TRUE)
SELECT {CLIENT} AS client, {CONTENT} AS content,
  SUM(CASE WHEN CAST({DATE} AS DATE) <= DATE '{CUT}' THEN {CLICKS} END) AS clicks_21d,
  SUM(CASE WHEN CAST({DATE} AS DATE) <= DATE '{CUT}' THEN {IMPR} END)   AS impressions_21d,
  AVG(CASE WHEN CAST({DATE} AS DATE) <= DATE '{CUT}' THEN {POS} END)    AS avg_position_21d,
  COUNT(DISTINCT CASE WHEN CAST({DATE} AS DATE) <= DATE '{CUT}' AND {IMPR} > 0 THEN {DATE} END) AS active_days_21d,
  SUM(CASE WHEN CAST({DATE} AS DATE) BETWEEN DATE '{MONTH}-15' AND DATE '{CUT}' THEN {CLICKS} END) AS clicks_days15_21,
  SUM(CASE WHEN CAST({DATE} AS DATE) >  DATE '{CUT}' THEN {CLICKS} END) AS label_clicks_next10d
FROM a GROUP BY 1, 2""").df()

feat = feat.fillna({"clicks_21d": 0, "impressions_21d": 0, "active_days_21d": 0,
                    "clicks_days15_21": 0, "label_clicks_next10d": 0})
feat = feat[feat.impressions_21d > 0].reset_index(drop=True)   # only content visible in the window
FEATS = ["clicks_21d", "impressions_21d", "avg_position_21d", "active_days_21d", "clicks_days15_21"]
print(feat.shape)
feat[FEATS + ["label_clicks_next10d"]].describe().round(2)   # no id columns shown

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(162301, 8)


,clicks_21d,impressions_21d,avg_position_21d,active_days_21d,clicks_days15_21,label_clicks_next10d
count,162301.00,162301.00,162301.00,162301.00,162301.00,162301.00
mean,3.35,1103.96,661.37,14.58,1.14,1.69
std,17.97,3559.24,3010.90,7.46,6.47,10.54
min,0.00,1.00,0.00,1.00,0.00,0.00
25%,0.00,18.00,23.50,8.00,0.00,0.00
50%,0.00,138.00,105.14,19.00,0.00,0.00
75%,2.00,768.00,365.19,21.00,0.00,1.00
max,3398.00,273012.00,219732.14,21.00,1277.00,2270.00


The trap: I add ONE label-derived column on purpose (the label divided by 10). The quick score jumps toward perfect, which is leakage, not skill. I then delete it and keep the honest number.

In [26]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

y = np.log1p(feat.label_clicks_next10d)
tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=0).split(feat, groups=feat.client))

def score(cols):
    mdl = HistGradientBoostingRegressor(random_state=0).fit(feat.iloc[tr][cols], y.iloc[tr])
    p = mdl.predict(feat.iloc[te][cols])
    return {"R2": round(r2_score(y.iloc[te], p), 3), "Spearman": round(spearmanr(y.iloc[te], p)[0], 3)}

honest = score(FEATS)
print("honest      :", honest)

feat["LEAK_next10d_per_day"] = feat.label_clicks_next10d / 10      # label-derived ON PURPOSE
print("with leak   :", score(FEATS + ["LEAK_next10d_per_day"]))    # jumps toward perfect

feat = feat.drop(columns="LEAK_next10d_per_day")                    # removed
print("honest again:", score(FEATS), "<- the number I keep")

honest      : {'R2': 0.597, 'Spearman': np.float64(0.457)}
with leak   : {'R2': 1.0, 'Spearman': np.float64(0.999)}
honest again: {'R2': 0.597, 'Spearman': np.float64(0.457)} <- the number I keep


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: my slice is one month (March 2026) of an unbalanced panel. Clients have different history depths, and early rows can be search-console-only, so what I observed here is directional and not guaranteed to hold for other months or clients.
Other limits:
- The label is a 10-day clicks proxy, a measured count, not a causal measure of content quality.
- Features (days 1-21) and label (days 22-31) do not overlap by design, but one month cannot show seasonality.
- Hashed ids mean I cannot add outside knowledge about any client or page.
- Results are decision-support signals, not guarantees.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Back the limits with aggregate numbers only (no ids printed)
print("content rows in feature frame:", len(feat))
print("clients in frame:", feat.client.nunique())
print("share of rows with zero label clicks:", round((feat.label_clicks_next10d == 0).mean(), 3))
print("largest single client share of rows:", round(feat.client.value_counts(normalize=True).iloc[0], 3))
print("honest score kept:", honest)

content rows in feature frame: 162301
clients in frame: 45
share of rows with zero label clicks: 0.73
largest single client share of rows: 0.163
honest score kept: {'R2': 0.597, 'Spearman': np.float64(0.457)}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.